# Data Generation for Synthetic RCT Diagnostics

This notebook explains not just how synthetic data is created, but what the stored dataset actually looks like and how to inspect it.

The project treats synthetic data as a reusable on-disk artifact rather than an in-memory side effect. The standard workflow is:

1. Check whether `data/synthetic_experiments.jsonl` already exists.
2. If it exists, load it.
3. If it does not exist, generate it once and save it.

That means notebooks, scripts, and evaluation runs all read from the same shared dataset by default.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from rct_diagnosis_agent.data import DEFAULT_DATASET_PATH, ensure_dataset, load_experiments

In [ ]:
dataset_path = ensure_dataset(dataset_path=DEFAULT_DATASET_PATH, count=25, seed=7)
experiments = load_experiments(dataset_path)
dataset_path, len(experiments)

## What is stored in the dataset

Each line in the JSONL file is one `ExperimentSummary` record. Each record contains compact experiment-level aggregates rather than raw user logs.

Key fields include:

- experiment identifiers and hypothesis text
- control and treatment group sizes
- pre-period and post-period means
- pre-period and post-period standard deviations
- control and treatment conversion rates
- optional segment summaries for Simpson's paradox scenarios
- hidden ground-truth issue labels used only for evaluation

This is enough structure for the agent to reason about common RCT pathologies without requiring row-level event data.

In [ ]:
experiments[0].model_dump()

## Converting the dataset to an analysis table

Working with a DataFrame makes it easier to inspect patterns across the stored experiments. We keep the full experiment objects for agent runs, but we also create a compact tabular view for exploratory analysis.

In [ ]:
rows = []
for exp in experiments:
    rows.append(
        {
            "experiment_id": exp.experiment_id,
            "control_size": exp.control_size,
            "treatment_size": exp.treatment_size,
            "total_size": exp.control_size + exp.treatment_size,
            "control_pre_mean": exp.control_pre_mean,
            "treatment_pre_mean": exp.treatment_pre_mean,
            "control_post_mean": exp.control_post_mean,
            "treatment_post_mean": exp.treatment_post_mean,
            "control_pre_std": exp.control_pre_std,
            "treatment_pre_std": exp.treatment_pre_std,
            "control_post_std": exp.control_post_std,
            "treatment_post_std": exp.treatment_post_std,
            "control_conversion_rate": exp.control_conversion_rate,
            "treatment_conversion_rate": exp.treatment_conversion_rate,
            "observed_conversion_diff": exp.treatment_conversion_rate - exp.control_conversion_rate,
            "pre_period_gap": exp.treatment_pre_mean - exp.control_pre_mean,
            "has_segments": bool(exp.segment_summaries),
            "issue_count": len(exp.hidden_truth),
            "hidden_truth": list(exp.hidden_truth),
            "notes": " | ".join(exp.notes),
        }
    )

df = pd.DataFrame(rows)
df.head()

## Schema-oriented spot check

This smaller view is useful when you want to quickly understand what kinds of values the stored summaries contain without scrolling through the full nested payload.

In [ ]:
df[[
    "experiment_id",
    "total_size",
    "pre_period_gap",
    "observed_conversion_diff",
    "has_segments",
    "issue_count",
    "hidden_truth",
]].head(10)

## Failure modes represented in the stored dataset

The generator can inject several common experiment issues:

- `srm`
- `pre_period_imbalance`
- `low_statistical_power`
- `outliers`
- `simpsons_paradox`

Because the dataset is saved to disk, the exact same synthetic experiments can be reused later for agent walkthroughs and evaluation.

In [ ]:
issue_counts = Counter(issue for exp in experiments for issue in exp.hidden_truth)
pd.Series(issue_counts).sort_values(ascending=False)

## How many issues appear per experiment?

Some summaries are relatively clean, while others combine multiple pathologies. That matters because the agent is expected to reason about overlapping explanations, not only isolated single-failure cases.

In [ ]:
df["issue_count"].value_counts().sort_index()

## Distribution summaries

Before plotting, it helps to look at a quick descriptive summary of the main numeric fields. This gives a compact sense of the range of sample sizes, mean shifts, and variance levels represented in the stored dataset.

In [ ]:
df[[
    "total_size",
    "control_conversion_rate",
    "treatment_conversion_rate",
    "observed_conversion_diff",
    "pre_period_gap",
    "control_post_std",
    "treatment_post_std",
]].describe().T

## Sample-size distribution

Low-power scenarios in this project are partly expressed through smaller effective sample sizes. Looking at the total-size distribution helps show whether the dataset contains a healthy spread from weaker to stronger experiments.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["total_size"], bins=10, color="#4C78A8", edgecolor="white")
plt.title("Distribution of total experiment sample size")
plt.xlabel("Total sample size")
plt.ylabel("Number of experiments")
plt.show()

## Observed conversion-lift distribution

The generator stores both control and treatment conversion rates. Their difference is a simple proxy for the observed treatment effect. This distribution helps illustrate that some experiments have clear directional signals while others are much closer to noise.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["observed_conversion_diff"], bins=10, color="#54A24B", edgecolor="white")
plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
plt.title("Distribution of observed treatment-control conversion difference")
plt.xlabel("Treatment conversion rate - Control conversion rate")
plt.ylabel("Number of experiments")
plt.show()

## Pre-period balance distribution

Pre-period imbalance is represented as a difference between treatment and control before the intervention. Plotting that gap across stored experiments makes the imbalance injection visible at the dataset level.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["pre_period_gap"], bins=10, color="#F58518", edgecolor="white")
plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
plt.title("Distribution of pre-period treatment-control mean gap")
plt.xlabel("Treatment pre mean - Control pre mean")
plt.ylabel("Number of experiments")
plt.show()

## Variance profile and outlier risk

Outlier-heavy summaries are represented by inflated post-period standard deviations. A simple scatter plot comparing pre-period and post-period variability makes those experiments easier to spot.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df["control_pre_std"], df["control_post_std"], label="Control", alpha=0.7, color="#4C78A8")
plt.scatter(df["treatment_pre_std"], df["treatment_post_std"], label="Treatment", alpha=0.7, color="#E45756")
max_std = max(df["control_post_std"].max(), df["treatment_post_std"].max())
plt.plot([0, max_std], [0, max_std], linestyle="--", color="black", linewidth=1)
plt.title("Pre vs post standard deviation")
plt.xlabel("Pre-period standard deviation")
plt.ylabel("Post-period standard deviation")
plt.legend()
plt.show()

## Inspecting one outlier-heavy experiment

The scatter plot gives a dataset-level view. Looking at one concrete example makes the outlier encoding more intuitive.

In [ ]:
outlier_example = next(exp for exp in experiments if "outliers" in exp.hidden_truth)
labels = ["control_pre", "control_post", "treatment_pre", "treatment_post"]
values = [
    outlier_example.control_pre_std,
    outlier_example.control_post_std,
    outlier_example.treatment_pre_std,
    outlier_example.treatment_post_std,
]

plt.figure(figsize=(8, 4))
plt.bar(labels, values, color=["#4C78A8", "#F58518", "#54A24B", "#E45756"])
plt.title("Variance profile for a stored outlier-heavy synthetic experiment")
plt.ylabel("Standard deviation")
plt.xticks(rotation=15)
plt.show()

## Segment-bearing experiments

Simpson's paradox cases are represented through `segment_summaries`. These records show how aggregate results can disagree with the segment-level story when the mix of users shifts between control and treatment.

In [ ]:
segment_examples = [exp for exp in experiments if exp.segment_summaries]
len(segment_examples), segment_examples[0].segment_summaries[0].model_dump() if segment_examples else None

## Why this dataset design is useful for the agent

The stored summaries are intentionally compact, but they preserve the signals the agent needs to reason about:

- allocation irregularities through group sizes
- pre-period imbalance through pre means and standard deviations
- low power through sample size and weak observed lift
- outlier risk through variance inflation
- Simpson's paradox through segment summaries

That keeps the educational project readable while still giving the agent enough structure to demonstrate tool use and reflection.